# 03.4 — K-fold stratification deep dive

Cross-validation splits trials into K folds; **stratified** cross-validation makes each fold representative of the whole — so a fold isn't accidentally all-easy-trials or all-one-session. This project ports MATLAB's *recursive* stratifier, which builds strata from a hierarchy of trial attributes. It's the most algorithm-heavy notebook in Module 03, and the payoff is understanding a real parity-tested port end to end.

This notebook covers:

* Why naive random K-fold is dangerous for imbalanced neural data.
* The recursive splitting idea, built from scratch on a tiny example.
* The production `stratify` + `assign_folds` pair, and what each guarantees.
* The honest parity boundary: what matches MATLAB and what can't.

**Prerequisites:** [01.2 control flow](../01_python_for_matlab_users/01.2_control_flow.ipynb) (recursion), a little [pandas]—but the DataFrame use here is light and explained inline.

## Section 1 — What MATLAB does

`cgg_getKFoldPartitions.m` builds folds by recursively partitioning trials along a **hierarchy** of attributes (`PARAMETERS_cgg_procSimpleDecoders_v2.m` defines the default 7-level hierarchy: outcome, dimensionality, gain, loss, …):

```matlab
% Sketch of the recursion:
function groups = splitRecursively(trials, levels)
    if numel(trials) <= NumFolds || isempty(levels)
        groups = {trials};                       % leaf — too small to split further
    else
        cats = unique(trials(:, levels{1}));      % split on this level's attribute(s)
        groups = {};
        for c = cats'
            sub = trials(trials(:, levels{1}) == c, :);
            groups = [groups, splitRecursively(sub, levels(2:end))];
        end
    end
end
```

Then `cvpartition` distributes the leaf groups across folds. The invariant MATLAB cares about: **the strata (leaf groups) are identical run to run** — only which fold each lands in depends on RNG.

## Section 2 — The Python concepts you need

### 2.1 — Why not just random K-fold?

Neural decoding data is imbalanced — most trials are correct, a few are errors; some sessions are large, some tiny. Random splitting can hand one fold almost no error trials, making its validation accuracy meaningless. Watch it happen on a deliberately imbalanced set:

In [ ]:
import numpy as np

rng = np.random.default_rng(4)
n = 60
outcome = np.array([1] * 54 + [0] * 6)      # 90% correct, 10% error — realistic imbalance
rng.shuffle(outcome)

# Naive random 5-fold: just chop the shuffled indices
folds_naive = np.array([i % 5 for i in range(n)])
rng.shuffle(folds_naive)
print("error-trial count per fold (naive random):")
for f in range(5):
    print(f"  fold {f}: {int((outcome[folds_naive == f] == 0).sum())} errors of {int((folds_naive==f).sum())}")

Some folds get 0 or 1 error trials — validating "error decoding" on a fold with no errors is noise. Stratification exists to prevent exactly this.

### 2.2 — Recursive splitting, from scratch

The core idea in ~10 lines of Python — split a group by an attribute; stop when a group is too small to split safely (≤ K trials, since you can't spread fewer than K trials across K folds evenly):

In [ ]:
def split_recursively(indices, level_values, num_folds):
    """Yield leaf groups: recursively split `indices` by each attribute level."""
    if len(indices) <= num_folds or not level_values:
        yield list(indices)                          # leaf
        return
    this_level = level_values[0]
    for category in np.unique(this_level[indices]):
        sub = indices[this_level[indices] == category]
        yield from split_recursively(sub, level_values[1:], num_folds)

# Toy data: 20 trials, two attributes (outcome, difficulty)
idx = np.arange(20)
outcome    = np.array([1,1,1,1,0,0, 1,1,1,1,1,1, 0,0,1,1,1,1,1,1])
difficulty = np.array([2,2,3,3,2,3, 2,2,3,3,2,3, 2,3,2,2,3,3,2,3])

leaves = list(split_recursively(idx, [outcome, difficulty], num_folds=5))
print(f"{len(leaves)} leaf groups:")
for g in leaves:
    print(f"  {g}  → outcome {set(outcome[g])}, difficulty {set(difficulty[g])}")

Each leaf is homogeneous in the attributes it was split on (all same outcome, all same difficulty) — those are the **strata**. Distributing whole leaves across folds then keeps each attribute-combination spread out rather than clumped.

### 2.3 — The production pair: `stratify` then `assign_folds`

The real code splits the two responsibilities cleanly — `stratify` produces stratum IDs (the deterministic, MATLAB-matched part), `assign_folds` distributes them (the RNG-dependent part):

In [ ]:
import pandas as pd
from neural_data_decoding.data.stratification import stratify, assign_folds

# 200 trials, two stratification attributes
rng = np.random.default_rng(0)
N = 200
trials = pd.DataFrame({
    "DataNumber":     range(1, N + 1),        # unique per-trial ID (required)
    "Outcome":        rng.integers(0, 2, N),
    "Dimensionality": rng.integers(1, 4, N),
})

# stratify: hierarchy = split on Outcome, then Dimensionality
group_ids = stratify(trials, all_split_names=[["Outcome"], ["Dimensionality"]], num_folds=5)
print("distinct strata:", len(set(group_ids.tolist())))

# assign_folds: distribute whole strata round-robin across folds
fold_ids = assign_folds(group_ids, num_folds=5, seed=0)
print("fold sizes:", [int((fold_ids == f).sum()) for f in range(1, 6)])

### 2.4 — What the guarantee actually is (read this carefully)

It's tempting to expect "every fold has the same class balance." The honest statement is subtler, and worth getting right:

* **`stratify` keeps each stratum whole** — a group of trials sharing an attribute-combination is never split across folds. This is the deterministic, MATLAB-parity-tested piece.
* **`assign_folds` distributes strata round-robin** in shuffled order. With *many small* strata (real data: hundreds of trials → dozens of leaves), fold sizes and composition come out well-balanced. With *few large* strata, a single big leaf lands entirely in one fold and balance suffers.

So the mechanism doesn't force per-fold balance; it makes balance the *expected* outcome when strata are numerous and small — which the deep 7-level hierarchy is designed to produce. Let's see the difference empirically:

In [ ]:
def balance_report(n_attr_levels, label):
    rng = np.random.default_rng(0)
    N = 300
    cols = {"DataNumber": range(1, N + 1), "Outcome": rng.integers(0, 2, N)}
    names = [["Outcome"]]
    for a in range(n_attr_levels):
        cols[f"Attr{a}"] = rng.integers(0, 3, N)
        names.append([f"Attr{a}"])
    df = pd.DataFrame(cols)
    g = stratify(df, all_split_names=names, num_folds=5)
    f = assign_folds(g, num_folds=5, seed=0)
    fracs = [df["Outcome"].values[f == k].mean() for k in range(1, 6)]
    spread = max(fracs) - min(fracs)
    print(f"{label}: {len(set(g.tolist()))} strata, "
          f"per-fold outcome-1 fraction spread = {spread:.2f} (overall {df.Outcome.mean():.2f})")

balance_report(0, "shallow hierarchy (few big strata)  ")
balance_report(4, "deep hierarchy (many small strata)  ")

The deep hierarchy produces far more strata and a tighter per-fold spread — quantitative confirmation of why the production default uses a 7-level hierarchy. This is exactly the kind of claim the curriculum insists on *showing* rather than asserting.

## Section 3 — The `neural_data_decoding` implementation

The recursive splitter core — compare it to your from-scratch version in §2.2:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/stratification.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def _split_recursively"))
for k in range(i, min(i + 26, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The `<= num_folds` leaf condition — the same "too small to split" guard, citing the reason in a comment.
* Group IDs are assigned in traversal order and start at 1 (MATLAB convention) — the docstring pins this as the parity invariant.
* The RNG-dependent fold assignment is deliberately a *separate* function (`assign_folds`), with a docstring stating plainly that it **cannot** bit-match MATLAB's `cvpartition` (different RNG) — so the parity tests assert on the strata, not the fold labels. Honest boundary-drawing is itself the lesson: know which parts of a port can match and which can't.

## Section 4 — Hands-on exercises

### Exercise 4.1 — recursion depth

Take the §2.2 toy data and split on `[difficulty]` ALONE (one level) vs `[outcome, difficulty]` (two levels). How many leaves each, and why? Predict before running.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
one = list(split_recursively(idx, [difficulty], num_folds=5))
two = list(split_recursively(idx, [outcome, difficulty], num_folds=5))
print(f"[difficulty] alone:      {len(one)} leaves")
print(f"[outcome, difficulty]:   {len(two)} leaves — finer, because each difficulty group")
print("                         is further split by outcome")

### Exercise 4.2 — the imbalance fix, measured

Repeat §2.1's error-count-per-fold, but this time use `stratify(..., [['Outcome']], num_folds=5)` + `assign_folds`. Do error trials spread more evenly? (With only 6 errors and one attribute, perfection isn't guaranteed — but observe the direction.)

In [ ]:
# Reveal:
outcome_60 = np.array([1] * 54 + [0] * 6)
np.random.default_rng(4).shuffle(outcome_60)          # shuffle the array BEFORE the DataFrame owns it
df = pd.DataFrame({"DataNumber": range(1, 61), "Outcome": outcome_60})
g = stratify(df, all_split_names=[["Outcome"]], num_folds=5)
f = assign_folds(g, num_folds=5, seed=1)
print("error-trial count per fold (stratified on Outcome):")
for k in range(1, 6):
    print(f"  fold {k}: {int((df['Outcome'].values[f == k] == 0).sum())} errors")
print("all 6 errors landed in ONE fold — with a single coarse attribute and only 6 errors,")
print("the stratum 'error' is one indivisible leaf. This is §2.4's few-large-strata case,")
print("and exactly why the production hierarchy is 7 levels deep.")

## Section 5 — Common errors

### `ValueError: data_number_column ... duplicate values`

`stratify` requires a unique per-trial ID. Duplicate `DataNumber`s mean two trials look identical to the splitter — deduplicate or fix the ID source.

### `KeyError` naming a split column

An attribute in `all_split_names` isn't a column of the identifiers DataFrame. Column names must match exactly (case-sensitive).

### Folds look unbalanced despite stratifying

Too few strata (§2.4) — a shallow hierarchy over coarse attributes. Add levels, or accept that a rare attribute-combination may cluster. Check `len(set(group_ids))` — if it's small relative to `num_folds`, that's your answer.

### Python fold labels don't match MATLAB's

Expected and documented — `assign_folds` can't reproduce MATLAB's `cvpartition` RNG. The *strata* match; the fold-label assignment doesn't. Parity tests know this.

### A stratum bigger than N/K trials

One leaf exceeds a fold's worth, so whichever fold gets it is oversized. Usually means an attribute is near-constant (splitting on it yields one giant group). Drop that level or add a finer one below it.

## Section 6 — Further reading

- [scikit-learn StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html) — the standard single-label stratifier, for contrast with this multi-level recursive one.
- [`src/neural_data_decoding/data/stratification.py`](../../src/neural_data_decoding/data/stratification.py) — the full port, docstrings first.
- [`tests/unit/test_stratification.py`](../../tests/unit/test_stratification.py) — the parity tests, showing exactly which invariant is asserted.

Next up: **[03.5 normalization recipes](03.5_normalization_recipes.ipynb)** — the string-driven dispatch that turns `"Channel - Z-Score - ..."` into a transform.